# 10-тарау. Токенизация: BPE-ді нөлден жазамыз

*«Машина қалай үйренеді? 2-том» кітабына арналған блокнот. Әр ұяшықтың түсіндірмесі кітапта берілген.*

Ұяшықтарды жоғарыдан төмен қарай ретімен іске қосыңыз: **Shift + Enter**.

## 10.1. Мәтінді қалай бөлшектейміз?

## 10.2. BPE алгоритмі

## 10.3. Мәтінді дайындау

In [ ]:
import re
import urllib.request
from collections import Counter

RAW = ("https://raw.githubusercontent.com/amirtemir1996n3-dotcom/"
       "mashina-qalai-uirenedi/main/")
urllib.request.urlretrieve(RAW + "data/kitap.txt", "kitap.txt")
text = open("kitap.txt", encoding="utf-8").read()
print(len(text), "таңба")
print(text[:120])

In [ ]:
def pre_tokenize(text):
    pieces = re.findall(r" ?\w+| ?[^\w\s]+|\s+", text)
    return [p.replace(" ", "▁") for p in pieces]

word_freq = Counter(pre_tokenize(text))
print("Бірегей сөздер:", len(word_freq))
print(word_freq.most_common(8))

## 10.4. BPE-ді оқытамыз

In [ ]:
def count_pairs(vocab):
    pairs = Counter()
    for symbols, freq in vocab.items():
        for a, b in zip(symbols, symbols[1:]):
            pairs[a, b] += freq
    return pairs

def merge_pair(vocab, pair):
    a, b = pair
    new_vocab = {}
    for symbols, freq in vocab.items():
        out, i = [], 0
        while i < len(symbols):
            if (i + 1 < len(symbols) and symbols[i] == a
                    and symbols[i + 1] == b):
                out.append(a + b)
                i += 2
            else:
                out.append(symbols[i])
                i += 1
        new_vocab[tuple(out)] = freq
    return new_vocab

def train_bpe(word_freq, num_merges):
    vocab = {tuple(w): f for w, f in word_freq.items()}
    merges = []
    for _ in range(num_merges):
        pairs = count_pairs(vocab)
        best = max(pairs, key=pairs.get)
        vocab = merge_pair(vocab, best)
        merges.append(best)
    return merges

merges = train_bpe(word_freq, 500)
print("Алғашқы біріктірулер:", merges[:8])
print("Соңғы біріктірулер:", merges[-6:])

In [ ]:
chars = sorted(set("".join(word_freq)))
tokens = chars + [a + b for a, b in merges]
token_to_id = {t: i for i, t in enumerate(tokens)}
print("Сөздік өлшемі:", len(tokens))

## 10.5. Кодтау және декодтау

In [ ]:
rank = {pair: i for i, pair in enumerate(merges)}

def encode_word(word):
    symbols = list(word)
    while len(symbols) > 1:
        pairs = [(rank.get(p, 10**9), i)
                 for i, p in enumerate(zip(symbols, symbols[1:]))]
        best_rank, i = min(pairs)
        if best_rank == 10**9:            # біріктіруге болатын жұп жоқ
            break
        symbols[i:i + 2] = [symbols[i] + symbols[i + 1]]
    return symbols

def encode(text):
    return [token_to_id[t] for w in pre_tokenize(text)
            for t in encode_word(w)]

def decode(ids):
    return "".join(tokens[i] for i in ids).replace("▁", " ")

sentence = "Нейрондық желілер мәтінді токендерге бөледі."
ids = encode(sentence)
print([tokens[i] for i in ids])
print(ids)
print(decode(ids) == sentence)

In [ ]:
ids = encode(text)
print(f"{len(text)} таңба → {len(ids)} токен")
print(f"Бір токенде орта есеппен {len(text) / len(ids):.2f} таңба")
russian = "Нейронные сети делят текст на токены."
print(len(sentence), "таңба →", len(encode(sentence)), "токен")
print(len(russian), "таңба →", len(encode(russian)), "токен")

## 10.6. Байттық BPE

In [ ]:
for word in ["hello", "сәлем", "中文"]:
    b = word.encode("utf-8")
    print(f"{word!r}: {len(word)} таңба, {len(b)} байт: {list(b)}")